In [ ]:
# PART 1: Data Acquisition, Cleaning & Exploratory Analysis
# Importing required libraries

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

# Displaying plots inside notebook
%matplotlib inline

In [ ]:
# Loading the dataset

df = pd.read_csv("data/healthcare-dataset-stroke-data.csv")

# Displaying the first 5 rows
df.head()

In [ ]:
# Displaying dataset information

print("Shape of the dataset:")
print(df.shape)

print("\nColumn Data Types:")
print(df.dtypes)

In [ ]:
# Null Value Analysis
# Counting missing values
null_count = df.isnull().sum()

# Calculating percentage of missing values
null_percentage = (null_count / df.shape[0]) * 100

# Creating a table
null_table = pd.DataFrame({
    "Missing Values": null_count,
    "Percentage": null_percentage
})

print(null_table)

In [ ]:
# Columns with more than 20% missing values

more_than_20 = null_table[null_table["Percentage"] > 20]

print("Columns with more than 20% missing values:")
print(more_than_20)

In [ ]:
# Filling Missing Values using Median
# Filling Missing Values in numeric columns with Median
numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns

for col in numeric_columns:
    if df[col].isnull().sum() > 0:
        median_value = df[col].median()
        df[col] = df[col].fillna(median_value)

print("Missing values after imputation:")
print(df.isnull().sum())

In [ ]:
# Duplicate Detection
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

In [ ]:
# Removing Duplicate Rows
# Storing number of rows before removing duplicates
rows_before = df.shape[0]

# Removing duplicates
df = df.drop_duplicates()

# Storing number of rows after removing duplicates
rows_after = df.shape[0]

print("Rows before removing duplicates:", rows_before)
print("Rows after removing duplicates :", rows_after)
print("Duplicate rows removed:", rows_before - rows_after)

In [ ]:
# Checking Missing Values Again
# Missing value percentage after duplicate removal

null_percentage_after = (df.isnull().sum() / df.shape[0]) * 100

print("Missing Value Percentage After Removing Duplicates:")
print(null_percentage_after)

In [ ]:
# Memory Usage Before Data Type Conversion
memory_before = df.memory_usage(deep=True).sum()

print("Memory usage before conversion:", memory_before, "bytes")

In [ ]:
# Converting repetitive string columns to category

category_columns = [
    "gender",
    "ever_married",
    "work_type",
    "Residence_type",
    "smoking_status"
]

for col in category_columns:
    df[col] = df[col].astype("category")

memory_after = df.memory_usage(deep=True).sum()

print("Memory usage after conversion:", memory_after, "bytes")
print("Memory saved:", memory_before - memory_after, "bytes")

In [ ]:
# Descriptive Statistics
print("Descriptive Statistics:")
print(df.describe())

In [ ]:
# Skewness of Numeric Columns
skewness = df.select_dtypes(include=['int64', 'float64']).skew()

print("Skewness of Numeric Columns:")
print(skewness)

In [ ]:
# Finding the column with the highest absolute skewness

most_skewed = skewness.abs().idxmax()

print("Most Skewed Column:", most_skewed)
print("Skewness Value:", skewness[most_skewed])

In [ ]:
# Continuous numeric columns (excluding ID and target)

continuous_cols = ["age", "avg_glucose_level", "bmi"]

skewness = df[continuous_cols].skew()

print("Skewness of Continuous Numeric Columns:")
print(skewness)

most_skewed = skewness.abs().idxmax()

print("\nMost Skewed Continuous Column:", most_skewed)
print("Skewness Value:", skewness[most_skewed])

In [ ]:
# Outlier Detection using IQR
columns = ["avg_glucose_level", "bmi"]

for col in columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower) | (df[col] > upper)]

    print("=" * 50)
    print("Column:", col)
    print("Q1:", Q1)
    print("Q3:", Q3)
    print("IQR:", IQR)
    print("Lower Bound:", lower)
    print("Upper Bound:", upper)
    print("Number of Outliers:", len(outliers))

In [ ]:
# Line Plot
plt.figure(figsize=(10,5))

plt.plot(df.index, df["age"])

plt.title("Age Distribution by Row Index")
plt.xlabel("Row Index")
plt.ylabel("Age")

plt.savefig("plots/line_plot.png")

plt.show()

In [ ]:
# Bar Chart
plt.figure(figsize=(8,5))

df.groupby("work_type")["bmi"].mean().plot(kind="bar")

plt.title("Average BMI by Work Type")
plt.xlabel("Work Type")
plt.ylabel("Average BMI")

plt.savefig("plots/bar_chart.png")

plt.show()

In [ ]:
# Histogram
plt.figure(figsize=(8,5))

sns.histplot(df["avg_glucose_level"], bins=20)

plt.title("Distribution of Average Glucose Level")
plt.xlabel("Average Glucose Level")
plt.ylabel("Frequency")

plt.savefig("plots/histogram.png")

plt.show()

In [ ]:
# Scatter Plot
plt.figure(figsize=(8,5))

sns.scatterplot(
    data=df,
    x="age",
    y="avg_glucose_level"
)

plt.title("Age vs Average Glucose Level")
plt.xlabel("Age")
plt.ylabel("Average Glucose Level")

plt.savefig("plots/scatter_plot.png")

plt.show()

In [ ]:
# Box Plot
plt.figure(figsize=(8,5))

sns.boxplot(
    data=df,
    x="gender",
    y="bmi"
)

plt.title("BMI Distribution by Gender")
plt.xlabel("Gender")
plt.ylabel("BMI")

plt.savefig("plots/box_plot.png")

plt.show()

In [ ]:
# Correlation Matrix (Pearson)

# Selecting only numeric columns
numeric_df = df.select_dtypes(include=['int64', 'float64'])

# Computing correlation matrix
correlation_matrix = numeric_df.corr()

print("Pearson Correlation Matrix:")
print(correlation_matrix)

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(10,8))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Heatmap")

plt.savefig("plots/correlation_heatmap.png")

plt.show()

In [ ]:
# Highest Absolute Correlation Pair
corr = correlation_matrix.abs()

# Removing self-correlations
np.fill_diagonal(corr.values, 0)

# Finding highest correlation pair
highest_pair = corr.unstack().idxmax()
highest_value = corr.unstack().max()

print("Highest Correlated Pair:", highest_pair)
print("Correlation Value:", highest_value)

In [ ]:
# Mean vs Median Comparison
columns = ["avg_glucose_level", "bmi"]

print("Mean vs Median Comparison\n")

for col in columns:
    print(f"{col}")
    print("Mean   :", df[col].mean())
    print("Median :", df[col].median())
    print("-" * 40)

In [ ]:
# Spearman Correlation Matrix
spearman_corr = numeric_df.corr(method="spearman")

print("Spearman Correlation Matrix:")
print(spearman_corr)

In [ ]:
# Pearson vs Spearman Difference

pearson_corr = correlation_matrix

difference = (spearman_corr - pearson_corr).abs()

print("Absolute Difference Between Spearman and Pearson")
print(difference)

In [ ]:
# Top 3 Largest Differences
diff = difference.where(
    np.triu(np.ones(difference.shape), k=1).astype(bool)
)

top3 = (
    diff.stack()
        .sort_values(ascending=False)
        .head(3)
)

print("Top 3 Column Pairs:")
print(top3)

In [ ]:
# Grouped Aggregation
group_stats = df.groupby(
    "work_type",
    observed=False
)["bmi"].agg(["mean", "std", "count"])

print(group_stats)

In [ ]:
highest_mean = group_stats["mean"].max()
lowest_mean = group_stats["mean"].min()

ratio = highest_mean / lowest_mean

print("Highest Mean:", highest_mean)
print("Lowest Mean :", lowest_mean)
print("Ratio :", ratio)

In [ ]:
# Save Clean Dataset

df.to_csv("cleaned_data.csv", index=False)

print("cleaned_data.csv saved successfully!")